In [1]:
import open3d as o3d
import numpy as np
from sklearn.decomposition import PCA
import copy

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
# Helper function to create coordinate axes
def create_coordinate_frame(size=0.5, origin=[0, 0, 0]):
    frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=size)
    frame.translate(origin)
    return frame


pcd = o3d.io.read_point_cloud(
    # "/home/sha/Documents/mesh/MeshroomCache/ConvertSfMFormat/ebd642612b8b456855af6468c5b3ed3e2a943192/sfm.ply"
    "/home/sha/Documents/work/robohand_v2/colmap/data/dense1/fused.ply"
)
print(f"Points: {len(pcd.points)}")


orig_frame = create_coordinate_frame(size=0.5)
print("Original point cloud with coordinate frame:")
o3d.visualization.draw_geometries([pcd, orig_frame])

Points: 408427
Original point cloud with coordinate frame:


In [5]:
# Step 1: Pre-process the point cloud
# We'll use a segmentation approach based on clustering
# Compute normals for better clustering
pcd.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.5, max_nn=30))

# Perform DBSCAN clustering to separate objects
# Adjust eps and min_points based on the scale and density of your point cloud
labels = np.array(pcd.cluster_dbscan(eps=0.05, min_points=6))
max_label = labels.max()
print(f"Point cloud has {max_label + 1} clusters")

# Create a list of point clouds for each cluster
clusters = []
for i in range(max_label + 1):
    cluster_indices = np.where(labels == i)[0]
    if len(cluster_indices) < 100:  # Skip very small clusters
        continue
    
    cluster = pcd.select_by_index(cluster_indices)
    # color = [np.random.uniform(0.3, 1.0) for _ in range(3)]
    # cluster.paint_uniform_color(color)
    clusters.append(cluster)

print("Point cloud clusters:")
o3d.visualization.draw_geometries(clusters + [orig_frame])

Point cloud has 2742 clusters
Point cloud clusters:


In [6]:
# Step 2: Find the largest horizontal plane (likely the table)
# First combine all clusters back
combined_pcd = o3d.geometry.PointCloud()
for cluster in clusters:
    combined_pcd += cluster

# Segment the table plane
plane_model, inliers = combined_pcd.segment_plane(distance_threshold=0.01, ransac_n=3, num_iterations=1000)
[a, b, c, d] = plane_model
print(f"Table plane equation: {a:.2f}x + {b:.2f}y + {c:.2f}z + {d:.2f} = 0")

# Extract the table and objects
table = combined_pcd.select_by_index(inliers)
table.paint_uniform_color([1.0, 0, 0])  # Red color for table
objects = combined_pcd.select_by_index(inliers, invert=True)
objects.paint_uniform_color([0, 1.0, 0])  # Green color for objects

# Visualize the segmentation
print("Segmented point cloud (red=table, green=objects):")
o3d.visualization.draw_geometries([table, objects, orig_frame])

Table plane equation: -0.23x + 0.95y + 0.22z + -4.39 = 0
Segmented point cloud (red=table, green=objects):


In [7]:
# Step 1: Segment the table plane directly from the original point cloud
plane_model, plane_inliers = pcd.segment_plane(
    distance_threshold=0.01, ransac_n=3, num_iterations=1000
)
[a, b, c, d] = plane_model
print(f"Table plane equation: {a:.2f}x + {b:.2f}y + {c:.2f}z + {d:.2f} = 0")

# Extract the table and non-table points
table = pcd.select_by_index(plane_inliers)
table.paint_uniform_color([1.0, 0, 0])  # Red for table
non_table = pcd.select_by_index(plane_inliers, invert=True)

# Step 2: Now perform clustering only on the non-table points (more efficient)
# Compute normals for better clustering
non_table.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.5, max_nn=30)
)

# Perform DBSCAN clustering to separate objects
labels = np.array(non_table.cluster_dbscan(eps=0.02, min_points=5))
max_label = labels.max()
print(f"Non-table points have {max_label + 1} clusters")

# Create colored point clouds for visualization
all_objects = []
part_cluster = None
largest_cluster_size = 0
largest_cluster_idx = -1

# Process each cluster
for i in range(max_label + 1):
    cluster_indices = np.where(labels == i)[0]

    # Skip very small clusters
    if len(cluster_indices) < 250:
        continue

    # Keep track of the largest cluster (likely your part of interest)
    if len(cluster_indices) > largest_cluster_size:
        largest_cluster_size = len(cluster_indices)
        largest_cluster_idx = i

    # Create a point cloud for this cluster
    cluster = non_table.select_by_index(cluster_indices)

    # Assign a distinct color for visualization
    color = [np.random.uniform(0.3, 1.0) for _ in range(3)]
    cluster.paint_uniform_color(color)
    all_objects.append(cluster)

def display_inlier_outlier(cloud):
    # inlier_cloud = cloud.select_by_index(ind)
    # outlier_cloud = cloud.select_by_index(ind, invert=True)

    print("Showing outliers (red) and inliers (gray): ")
    # outlier_cloud.paint_uniform_color([1, 0, 0])
    # inlier_cloud.paint_uniform_color([0.8, 0.8, 0.8])
    o3d.visualization.draw_geometries([cloud])
    
def remove_outliers(pcd, nb_points=20, radius=2.0):
    # l, ind = voxel_down_pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
    return display_inlier_outlier(pcd)


filtered_objects = []
for obj in all_objects:
    # filtered_obj = remove_outliers(obj)
    # filtered_objects.append(filtered_obj)
    o3d.visualization.draw_geometries([obj])

Table plane equation: -0.23x + 0.95y + 0.22z + -4.40 = 0
Non-table points have 3881 clusters


In [6]:
# Step 4: Select the part of interest (either automatically or manually)
# Here we're using the largest cluster after filtering
part_cluster = None
largest_size = 0

for i, cluster in enumerate(all_objects):
    if len(cluster.points) > largest_size:
        largest_size = len(cluster.points)
        part_cluster = cluster

if part_cluster is not None:
    part_cluster.paint_uniform_color([0, 0, 1.0])  # Blue for part
    
    # Visualize table and part
    print("Table (red) and part (blue):")
    o3d.visualization.draw_geometries([table, part_cluster])

    # Step 5: Calculate rotation to align with XY-plane
    # Get the table normal and normalize it
    plane_normal = np.array([a, b, c])
    plane_normal = plane_normal / np.linalg.norm(plane_normal)
    print(f"Table normal: {plane_normal}")
    
    # Ensure normal points upward
    if plane_normal[2] < 0:
        plane_normal = -plane_normal
        print(f"Normal flipped to point upward: {plane_normal}")
    
    # Calculate rotation to align with Z-axis
    target = np.array([0, 0, 1])  # Z-axis (up)
    
    # Calculate rotation axis and angle
    rotation_axis = np.cross(plane_normal, target)
    rotation_axis_norm = np.linalg.norm(rotation_axis)
    
    if rotation_axis_norm < 1e-6:
        # Vectors are nearly parallel
        dot_product = np.dot(plane_normal, target)
        if dot_product > 0:
            # Same direction, no rotation needed
            R = np.eye(3)
            print("No rotation needed - normal already points up")
        else:
            # Opposite direction, rotate 180 degrees around x-axis
            R = np.array([
                [1, 0, 0],
                [0, -1, 0],
                [0, 0, -1]
            ])
            print("Normal points opposite to up - flipping")
    else:
        # Calculate rotation using Rodrigues formula
        rotation_axis = rotation_axis / rotation_axis_norm
        cos_angle = np.dot(plane_normal, target)
        angle = np.arccos(np.clip(cos_angle, -1.0, 1.0))
        
        K = np.array([
            [0, -rotation_axis[2], rotation_axis[1]],
            [rotation_axis[2], 0, -rotation_axis[0]],
            [-rotation_axis[1], rotation_axis[0], 0]
        ])
        R = np.eye(3) + np.sin(angle) * K + (1 - np.cos(angle)) * (K @ K)
        
        print(f"Rotating by {np.degrees(angle):.2f} degrees around axis {rotation_axis}")
    
    print(f"Rotation matrix:\n{R}")
    
    # Step 6: Apply rotation to align table with XY-plane
    aligned_table = copy.deepcopy(table)
    aligned_table.rotate(R)
    
    aligned_part = copy.deepcopy(part_cluster)
    aligned_part.rotate(R)
    
    # Step 7: Center the point cloud horizontally but keep vertical position
    table_center = aligned_table.get_center()
    translation = [-table_center[0], -table_center[1], 0]  # Only center X and Y
    
    aligned_table.translate(translation)
    aligned_part.translate(translation)
    
    # Find the Z-coordinate of the table surface
    aligned_table_points = np.asarray(aligned_table.points)
    table_z = np.mean(aligned_table_points[:, 2])
    print(f"Table surface Z-coordinate: {table_z:.4f}")
    
    # Create aligned coordinate frame at the table surface level
    aligned_frame = create_coordinate_frame(size=0.5, origin=[0, 0, table_z])
    
    # Visualize aligned part and table with coordinate frame
    print("Aligned point cloud with coordinate frame at table level:")
    o3d.visualization.draw_geometries([aligned_table, aligned_part, aligned_frame])

    # Step 9: Export the results (optional)
    o3d.io.write_point_cloud("aligned_part.ply", part_cluster)
    
    print("Results saved.")
else:
    print("No significant clusters found after filtering.")

Table (red) and part (blue):
Table normal: [ 0.16903031  0.89570638 -0.41126492]
Normal flipped to point upward: [-0.16903031 -0.89570638  0.41126492]
Rotating by 65.72 degrees around axis [-0.98265583  0.1854387   0.        ]
Rotation matrix:
[[ 0.97975487 -0.10728073  0.16903031]
 [-0.10728073  0.43151006  0.89570638]
 [-0.16903031 -0.89570638  0.41126492]]
Table surface Z-coordinate: -0.5302
Aligned point cloud with coordinate frame at table level:
Results saved.


In [8]:
import json


aligned_part_path = "/home/sha/Documents/work/robohand/vision/aligned_part.ply"
marker_data_path = "/home/sha/Documents/mesh/aruco_results/averaged_markers.json"
aligned_pcd = o3d.io.read_point_cloud(aligned_part_path)

def load_marker_positions(marker_data_path):
    with open(marker_data_path, 'r') as f:
        marker_data = json.load(f)
    return marker_data

def create_coordinate_system(size=0.1, origin=[0, 0, 0]):
    frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=size)
    frame.translate(origin)
    return frame

def transform_coordinate_system(coord_system, position, rotation_matrix):
    transformed = copy.deepcopy(coord_system)
    transformed.rotate(np.array(rotation_matrix))
    transformed.translate(position)
    return transformed
marker_data = load_marker_positions(marker_data_path)

marker_geometries = []
for marker_id, marker_info in marker_data.items():
    position = np.array(marker_info["position"])
    rotation = np.array(marker_info["rotation"])
    count = marker_info["count"]
    
    marker_frame = create_coordinate_system(size=0.05)
    marker_frame.rotate(rotation)
    marker_frame.translate(position)
    marker_geometries.append(marker_frame)
    
    marker_sphere = o3d.geometry.TriangleMesh.create_sphere(radius=0.01)
    marker_sphere.paint_uniform_color([1, 0, 1])
    marker_sphere.translate(position)
    marker_geometries.append(marker_sphere)
geometries = [aligned_pcd] + marker_geometries

o3d.visualization.draw_geometries(
    geometries, 
    zoom=0.7,
)

RPly: Unable to open file


[Open3D WARNING] Read PLY failed: unable to open file: /home/sha/Documents/work/robohand/vision/aligned_part.ply
[Open3D WARNING] The number of points is 0 when creating axis-aligned bounding box.
